# 🎓 第2章 第2课：三种回归 —— 时序 / 截面 / Fama-MacBeth

**本节目标**：先造一个“已知真相”的世界，再看三种方法能不能把真相挖回来。

对应原书 2.2 节。

- **时序回归** → 恢复 β（敏感度）；
- **截面回归** → 恢复 λ（因子身价）；
- **FM 回归** → 恢复 λ + 正确标准误。

> 口诀：**截面回归 = 先均值再回归；FM = 先回归再均值。**

In [1]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'src' / 'factor_zero').exists():
            return p
    raise FileNotFoundError('找不到仓库根目录，请从仓库内启动 notebook')
ROOT = _find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from factor_zero.sorting import quantile_group, sort_portfolios, spread_return, monotonicity_test
from factor_zero.factors import monthly_returns, momentum, book_to_market, log_market_cap
from factor_zero.statistics import nw_t_stat

def _find_sample():
    for p in [ROOT / 'data' / 'sample', Path('data/sample'), Path.cwd().parents[2] / 'data' / 'sample']:
        if p.exists():
            return p
    raise FileNotFoundError('找不到 data/sample，请从仓库内启动 notebook')
SAMPLE = _find_sample()
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
print('✅ 环境就绪，开始学习！')

from factor_zero.data_sim import simulate_cross_section
from factor_zero.regression import time_series_betas, cross_sectional_lambda, fama_macbeth

✅ 环境就绪，开始学习！


## 0) 造一个已知真相的世界

我们预设每只股票的 β 和因子溢价 λ，生成收益数据。
因为真相已知，才能判断方法准不准。

In [2]:
sim = simulate_cross_section(n_assets=200, t_periods=240, n_factors=3, noise=0.01, seed=7)
print('预设的真相 λ =', sim['lambda'].round(5))
print('收益矩阵:', sim['ret'].shape, '| 特征矩阵:', sim['char'].shape)

预设的真相 λ = [0.00881 0.00804 0.00469]
收益矩阵: (240, 200) | 特征矩阵: (240, 200, 3)


## 1) 时序回归：恢复 β

对每个资产：$r_{it} = lpha_i + eta_i' f_t + arepsilon_{it}$。
估计出的 β 应该接近预设值。

In [3]:
fdf = pd.DataFrame(sim['factor_ret'], columns=['F1', 'F2', 'F3'])
alphas, betas, resid = time_series_betas(sim['ret'], fdf)
err = np.abs(betas.to_numpy() - sim['beta']).max()
print(f'β 恢复最大误差: {err:.5f}（越小越准）')
betas.head(5).round(3)

β 恢复最大误差: 0.12111（越小越准）


,F1,F2,F3
a0,1.071,1.392,1.324
a1,0.739,0.800,1.415
a2,0.472,1.310,1.325
a3,0.976,0.792,0.839
a4,0.766,0.950,0.991


## 2) 截面回归：恢复 λ

第二步：平均收益 ~ β，斜率就是 λ。

注意：这里的 β 是**估计值**（生成回归变量），严格说需要 Shanken 修正（2.2.2 节）；
这里先用朴素 OLS 感受数量级。

In [4]:
mean_ret = sim['ret'].mean(axis=0)
lam_cs, alphas_cs = cross_sectional_lambda(betas, mean_ret)
print('截面回归 λ =', lam_cs.round(5))
print('预设真相 λ =', sim['lambda'].round(5))
print('误差        =', (lam_cs - sim['lambda']).round(5))

截面回归 λ = [0.00874 0.0082  0.0046 ]
预设真相 λ = [0.00881 0.00804 0.00469]
误差        = [-7.0e-05  1.6e-04 -9.0e-05]


## 3) Fama-MacBeth：恢复 λ + 标准误

每一期单独做截面回归，得到 T 个 λ_t，再平均——这就是“先回归、再均值”。
标准误用 Newey-West 修正。

In [5]:
lam_fm, lam_t, se, t_stat = fama_macbeth(sim['ret'], sim['char'])
res = pd.DataFrame({
    'FM λ': lam_fm[1:],
    '真相 λ': sim['lambda'],
    '标准误': se[1:],
    't 值': t_stat[1:],
}, index=['F1', 'F2', 'F3'])
res.round(4)

,FM λ,真相 λ,标准误,t 值
F1,0.0087,0.0088,0.0011,7.7524
F2,0.0080,0.0080,0.0014,5.8408
F3,0.0045,0.0047,0.0013,3.3487


In [6]:
print('每个因子 |t| > 2 → 在 95% 置信下显著')
print('全部显著 ✓' if (np.abs(t_stat[1:]) > 2).all() else '有不显著的因子 ✗')

每个因子 |t| > 2 → 在 95% 置信下显著
全部显著 ✓


## 4) 上 A 股快照：FM 检验价值 / 规模 / 动量

用第 2.1 节构造的因子（BM、对数市值、动量）当特征，跑 FM。
**注意**：这是教学面板，结果不代表真实 A 股结论；重点是看懂流程。

特征先做截面 z-score，便于比较系数大小。

In [7]:
def load_ashare_snapshot():
    """加载 87 只 A 股快照，构造教学用月度面板。

    真实数据：日线收盘价（2018-2024）+ 估值快照（2026-09-01）。
    估值快照只有一期；为了演示“逐期排序”，这里用价格路径 + 快照估值
    推导一个教学用月度基本面面板（合成，仅用于方法演示，不代表真实历史估值）。
    """
    daily_dir = SAMPLE / 'a_share_snapshot' / 'daily'
    codes = sorted(p.stem for p in daily_dir.glob('*.csv'))
    prices = {}
    for code in codes:
        df = pd.read_csv(daily_dir / f'{code}.csv', parse_dates=['date'])
        prices[code] = df.set_index('date')['close']
    price_df = pd.DataFrame(prices).sort_index()

    ret = monthly_returns(price_df).dropna(axis=1, how='any')
    fund = pd.read_csv(SAMPLE / 'a_share_snapshot' / 'fundamentals.csv', parse_dates=['trade_date'])
    fund['stock'] = fund['stock'].astype(int).astype(str).str.zfill(6)
    fund = fund.set_index('stock').reindex(ret.columns)

    month_end = price_df.resample('ME').last().ffill().reindex(columns=ret.columns)
    last_price = month_end.iloc[-1]
    shares = fund['total_mv'] / last_price
    total_mv = month_end.mul(shares, axis=1)

    base_pb = fund['pb']
    months = np.arange(len(month_end))
    phase = pd.Series(np.linspace(0, 2 * np.pi, len(ret.columns), endpoint=False), index=ret.columns)
    seasonal = pd.DataFrame(
        np.sin(months[:, None] / 12 * 2 * np.pi + phase.values[None, :]),
        index=month_end.index, columns=ret.columns,
    )
    pb = (1 + 0.15 * seasonal).mul(base_pb, axis=1).clip(lower=0.05)
    pe = pb / 0.10  # 教学用固定 ROE=10%，仅用于补全字段

    panel = pd.concat([
        pd.DataFrame({
            'stock': col, 'trade_date': month_end.index,
            'pb': pb[col].values, 'pe': pe[col].values, 'total_mv': total_mv[col].values,
        }) for col in ret.columns
    ], ignore_index=True)

    bm = book_to_market(panel).reindex(index=ret.index, columns=ret.columns)
    mv = log_market_cap(panel).reindex(index=ret.index, columns=ret.columns)
    mom = momentum(ret, lookback=12, skip=1).reindex(index=ret.index, columns=ret.columns)
    return {'ret': ret, 'bm': bm, 'mv': mv, 'mom': mom, 'price': price_df, 'panel': panel}

data = load_ashare_snapshot()
ret, bm, mv, mom = data['ret'], data['bm'], data['mv'], data['mom']
idx = ret.index.intersection(bm.index).intersection(mv.index).intersection(mom.index)
ret, bm, mv, mom = ret.loc[idx], bm.loc[idx], mv.loc[idx], mom.loc[idx]

def zscore(df):
    return df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0)

X = np.stack([zscore(bm), zscore(mv), zscore(mom)], axis=2)
X = np.nan_to_num(X, nan=0.0)  # 动量前 12 个月缺失 → 视为截面平均
lam_fm, lam_t, se, t_stat = fama_macbeth(ret, X)
res = pd.DataFrame({
    'λ': lam_fm[1:],
    '标准误': se[1:],
    't 值': t_stat[1:],
}, index=['价值(BM)', '规模(ln市值)', '动量'])
res.round(4)

,λ,标准误,t 值
价值(BM),-0.0040,0.0022,-1.8359
规模(ln市值),0.0039,0.0021,1.8447
动量,0.0029,0.0028,1.0188


## ✋ 试一试

1. 把 `simulate_cross_section` 的 `n_factors` 改成 5，看 FM 还能不能恢复全部 λ；
2. 用 `lambda_values=np.array([0.006, 0.004, 0.0])` 造一个“没被定价”的因子，看它的 t 值；
3. 把 A 股 FM 的 `mom` 换成滞后一期，结果变多少？

## ✅ 本节小结

- 三种回归的关系：**时序求 β → 截面求 λ**；FM = 截面回归的逐期版；
- FM 在已知真相数据上能恢复预设 λ（含标准误），方法可信；
- 真实数据上因子显著性会随样本、区间、数据处理变化——这就是第3章要做的事。

> 下一课：异象检验 + GRS + 正交化。